In [14]:
"""
Макропрогнозирование (Эконометрика против ML)
В рамках этого практического модуля мы решаем классическую задачу макроэкономического прогнозирования: 
предсказание будущих темпов роста экономики (ВВП) на горизонте одного года (4 квартала).

Главная цель этого задания — на практике столкнуть лбами два принципиально разных аналитических подхода:

Классическую эконометрику, к которой вы привыкли (модели векторной авторегрессии — VAR), где упор делается на 
совместную динамику и статистические свойства временных рядов.
Машинное обучение (ансамблевые модели: Random Forest и Gradient Boosting), где алгоритм самостоятельно ищет 
нелинейные зависимости и сложные паттерны в данных для максимизации точности прогноза.
Помимо самого моделирования, мы пройдем весь путь инженера данных (Data Engineer), максимально приближенный к реальным 
задачам в банках и хедж-фондах: от автоматического сбора сырых данных по API до их очистки и сведения в единую структуру с помощью SQL.

Какие данные мы используем?
Для построения прогноза мы будем использовать открытые данные Федерального резервного банка Сент-Луиса (база FRED). 
Это один из самых надежных и популярных источников макростатистики в мире.

Мы возьмем четыре ключевых индикатора, которые описывают разные аспекты экономики, но при этом имеют разную частотность публикации,
что создает типичную проблему для аналитика:

Реальный ВВП (GDP) — наш целевой показатель. Отражает общий объем экономики. Частотность: Квартальная.
Индекс потребительских цен (CPIAUCSL) — ключевая мера инфляции. Частотность: Месячная.
Уровень безработицы (UNRATE) — индикатор состояния рынка труда. Частотность: Месячная.
Доходность 10-летних казначейских облигаций США (DGS10) — индикатор стоимости денег и ожиданий инвесторов 
относительно будущих процентных ставок и инфляции. Частотность: Дневная.

Что именно будет происходить в коде (Краткий план действий):
Сбор и хранение: Сначала мы программно скачаем ряды разной частотности напрямую с серверов FRED и загрузим их в 
базу данных MySQL в "сыром" виде. На этом этапе данные будут полны пропусков (например, значение ВВП будет публиковаться 
лишь раз в квартал, а доходность облигаций — каждый день).
SQL-агрегация: С помощью SQL-запроса мы решим проблему частотности (Mixed-Frequency Data). Мы приведем все данные к единой 
квартальной сетке: ВВП возьмем по факту публикации, а дневные и месячные показатели (ставки, инфляцию, безработицу) усредним за соответствующий квартал.
Эконометрический этап: Мы проверим данные на стационарность, перейдем от абсолютных значений к темпам прироста и 
первым разностям (чтобы избежать ложной регрессии), а затем построим модель VAR(p) для прогнозирования на 4 квартала вперед.
Feature Engineering для ML: Поскольку деревья решений не понимают концепцию "времени", мы вручную создадим для них 
предикторы: сдвинем ряды в прошлое (добавим лаги) и посчитаем скользящую волатильность рынка. Также мы применим 
подход Direct Forecasting — сдвинем сам целевой ВВП на 4 квартала назад, чтобы модель училась предсказывать отдаленное будущее напрямую.
Обучение и оценка: Мы обучим Случайный лес и Градиентный бустинг, строго разделив выборку на обучающую (до 2019 года) 
и тестовую (с 2019 года). В конце мы сравним ошибку прогноза (RMSE и MAE) у эконометрики и ML-алгоритмов, а также посмотрим 
на график важности признаков, чтобы понять, на какие именно индикаторы опирался искусственный интеллект при принятии решений.
"""

''

In [3]:
!pip3 install fredapi

In [4]:
import pandas as pd
from fredapi import Fred
from sqlalchemy import create_engine

In [9]:
FRED_API_KEY = ''

TICKERS = {
    'gdp': 'GDP',
    'cpi': 'CPIAUCSL',
    'unrate': 'UNRATE',
    'dgs10': 'DGS10'
}

fred = Fred(api_key=FRED_API_KEY)
df_macro = pd.DataFrame()
for col_name, ticker in TICKERS.items():
    df_macro[col_name] = fred.get_series(ticker)

df_macro.reset_index(inplace=True)
df_macro.rename(columns={'index': 'date'}, inplace=True)
df_macro.head()

,date,gdp,cpi,unrate,dgs10
0,1946-01-01,NaN,NaN,NaN,NaN
1,1946-04-01,NaN,NaN,NaN,NaN
2,1946-07-01,NaN,NaN,NaN,NaN
3,1946-10-01,NaN,NaN,NaN,NaN
4,1947-01-01,243.164,21.48,NaN,NaN


In [1]:

# TODO 1.1: Заполните параметры подключения
DB_USER = '<ПОЛЬЗОВАТЕЛЬ>'
DB_PASSWORD = '<ПАРОЛЬ>'
DB_HOST = '127.0.0.1'
DB_NAME = 'macro_db'


fred = Fred(api_key=FRED_API_KEY)
df_macro = pd.DataFrame()

for col_name, ticker in TICKERS.items():
    df_macro[col_name] = fred.get_series(ticker)

df_macro.reset_index(inplace=True)
df_macro.rename(columns={'index': 'date'}, inplace=True)

# TODO 1.2: Отфильтруйте DataFrame, оставив только наблюдения начиная с 1 января 1990 года
df_macro = # <ВАШ_КОД>

conn_string = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
engine = create_engine(conn_string)

# TODO 1.3: Вызовите метод записи в SQL. 
# Используйте имя таблицы 'raw_macro_data'. 
# Настройте параметр if_exists так, чтобы таблица перезаписывалась, и отключите экспорт индекса.
# <ВАШ_КОД>


SyntaxError: invalid syntax (3854188217.py, line 29)

In [10]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

# TODO 2.1: Допишите SQL-запрос. 
# Нам нужно агрегировать данные по годам и кварталам.
# ВВП (gdp) нужно взять как максимальное значение (MAX) за квартал.
# Остальные переменные (cpi, unrate, dgs10) - как среднее (AVG).
query = """
SELECT 
    CONCAT(YEAR(date), '-', LPAD((QUARTER(date)-1)*3 + 1, 2, '0'), '-01') AS quarter_date,
    -- <ВАШ_КОД: агрегация gdp> AS gdp,
    -- <ВАШ_КОД: агрегация cpi> AS cpi,
    -- <ВАШ_КОД: агрегация unrate> AS unrate,
    -- <ВАШ_КОД: агрегация dgs10> AS dgs10
FROM raw_macro_data
GROUP BY YEAR(date), QUARTER(date)
ORDER BY quarter_date;
"""

df_quarterly = pd.read_sql(query, con=engine)
df_quarterly['quarter_date'] = pd.to_datetime(df_quarterly['quarter_date'])
df_quarterly.set_index('quarter_date', inplace=True)
df_quarterly.dropna(inplace=True) 

# TODO 2.2: Переведите ряды в стационарный вид.
# Для ВВП (gdp) и инфляции (cpi) рассчитайте темп прироста: разность логарифмов, умноженная на 100.
# Для ставок (unrate, dgs10) возьмите обычную первую разность (метод .diff() в pandas).
df_diff = pd.DataFrame()
df_diff['gdp_growth'] = # <ВАШ_КОД>
df_diff['cpi_inflation'] = # <ВАШ_КОД>
df_diff['unrate_diff'] = # <ВАШ_КОД>
df_diff['dgs10_diff'] = # <ВАШ_КОД>
df_diff.dropna(inplace=True)

# TODO 2.3: Обучите модель VAR.
# Инициализируйте модель, передав ей датафрейм df_diff.
# Используйте метод .fit() с выбором оптимального лага по критерию AIC (максимальный лаг задайте равным 4).
model_var = # <ВАШ_КОД>
results_var = # <ВАШ_КОД>
print("Оптимальный порядок VAR(p):", results_var.k_ar)

# TODO 2.4: Сделайте прогноз на 4 квартала вперед, используя метод .forecast()
forecast_var = # <ВАШ_КОД>


SyntaxError: invalid syntax (2446035134.py, line 31)

In [11]:
df_ml = df_diff.copy()

# TODO 3.1: Сформируйте целевую переменную (Direct Forecasting).
# Нам нужно предсказать рост ВВП через год (4 квартала). 
# Используйте сдвиг ряда назад во времени (метод .shift с отрицательным значением).
df_ml['target_gdp_4q'] = # <ВАШ_КОД>

# TODO 3.2: Сгенерируйте лаговые признаки.
# Создайте циклом лаги (t-1 и t-2) для всех предикторов из списка feature_cols.
feature_cols =['gdp_growth', 'cpi_inflation', 'unrate_diff', 'dgs10_diff']

for col in feature_cols:
    df_ml[f'{col}_lag1'] = # <ВАШ_КОД>
    df_ml[f'{col}_lag2'] = # <ВАШ_КОД>

# TODO 3.3: Добавьте оконную статистику (Rolling Window).
# Рассчитайте скользящее стандартное отклонение роста ВВП за последние 4 квартала (волатильность).
df_ml['gdp_volatility_4q'] = # <ВАШ_КОД>

df_ml.dropna(inplace=True)

# TODO 3.4: Разделите выборку на Train и Test без перемешивания (Data Leakage prevention).
# Обучающая выборка: строго до 1 января 2019 года.
# Тестовая выборка: начиная с 1 января 2019 года включительно.
train = # <ВАШ_КОД>
test = # <ВАШ_КОД>

X_cols =[c for c in df_ml.columns if 'target' not in c]

X_train, y_train = train[X_cols], train['target_gdp_4q']
X_test, y_test = test[X_cols], test['target_gdp_4q']


SyntaxError: invalid syntax (2478544970.py, line 6)

In [12]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# TODO 4.1: Инициализируйте RandomForestRegressor. 
# Задайте количество деревьев = 100, максимальную глубину = 5 и зафиксируйте random_state.
rf_model = # <ВАШ_КОД>

# TODO 4.2: Обучите модель на тренировочных данных и сделайте прогноз на тестовых.
# <ВАШ_КОД: Обучение>
rf_preds = # <ВАШ_КОД: Прогноз>

# TODO 4.3: Инициализируйте GradientBoostingRegressor.
# Параметры: 100 деревьев, скорость обучения (learning_rate) = 0.05, глубина = 3.
gb_model = # <ВАШ_КОД>

# TODO 4.4: Обучите модель градиентного бустинга и сделайте прогноз.
# <ВАШ_КОД: Обучение>
gb_preds = # <ВАШ_КОД: Прогноз>


SyntaxError: invalid syntax (164742816.py, line 5)

In [13]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error

models_predictions = {
    'Random Forest': rf_preds,
    'Gradient Boosting': gb_preds
}

# TODO 5.1: В цикле рассчитайте метрики качества (RMSE и MAE) для прогнозов.
# Выведите результаты на экран. Помните, что RMSE - это корень из mean_squared_error.
for name, preds in models_predictions.items():
    rmse = # <ВАШ_КОД>
    mae = # <ВАШ_КОД>
    print(f"[{name}] RMSE: {rmse:.4f} | MAE: {mae:.4f}")

# TODO 5.2: Извлеките важность признаков (Feature Importances) из обученной модели RandomForest.
# Передайте массив важностей в pd.Series, используя X_cols в качестве индекса.
importances_array = # <ВАШ_КОД: атрибут важности из rf_model>
feature_importances = pd.Series(importances_array, index=X_cols)

# Визуализация (оставьте как есть для экономии времени на уроке)
plt.figure(figsize=(12, 6))
plt.plot(y_test.index, y_test, label='Фактический рост ВВП (h=4)', marker='o')
plt.plot(y_test.index, rf_preds, label='Прогноз Random Forest', linestyle='--')
plt.plot(y_test.index, gb_preds, label='Прогноз Gradient Boosting', linestyle='-.')
plt.title('Прогнозирование темпов роста ВВП (Горизонт 4 квартала)')
plt.ylabel('Рост ВВП, %')
plt.grid(True)
plt.legend()
plt.show()

feature_importances.nlargest(10).sort_values().plot(kind='barh', figsize=(10, 5))
plt.title('Топ-10 значимых предикторов (Random Forest)')
plt.xlabel('Относительная важность')
plt.show()


SyntaxError: invalid syntax (912842766.py, line 12)